# Clasificar un volumen de OrganMNIST3D con la ResNet 3D entrenada (64×64×64)

Carga los pesos `modelo_seed0.pt` (semilla 0 de la corrida `OrganMNIST3D_64_Kaggle`, exactitud 0,957 en prueba),
toma un volumen del conjunto de **prueba** (pacientes que la red nunca vio), muestra sus tres cortes centrales
y dibuja las 11 probabilidades que da la red. Corre en CPU; no necesita GPU.

Qué hace la red con el volumen: escala a [0, 1] → normaliza con la media y la desviación del conjunto de entrenamiento →
convoluciones 3D (ResNet de 4 etapas) → promedio global → capa lineal de 11 salidas → softmax.
La clase predicha es la de mayor probabilidad.

Código, resultados y pesos: https://github.com/luis-122003/organmnist3d-2.5d-vs-3d

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "medmnist"], check=True)
import glob, hashlib, os, urllib.request
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

REPO_RAW = "https://github.com/luis-122003/organmnist3d-2.5d-vs-3d/raw/main/resultados_64/"
PESOS = "modelo_seed0.pt"
MD5_64 = "58a2205adf14a9d0a189cb06dc78bf10"
CLASES = ["hígado", "riñón der.", "riñón izq.", "fémur der.", "fémur izq.", "vejiga",
          "corazón", "pulmón der.", "pulmón izq.", "bazo", "páncreas"]

## 1. Pesos y datos

Busca `modelo_seed0.pt` en la carpeta actual o en `/kaggle/input`; si no está, lo descarga del repositorio. El dataset de 64³ (361 MB) se descarga con el paquete oficial `medmnist` desde Zenodo y se verifica su MD5.

In [ ]:
def buscar(nombre):
    for ruta in [nombre, f"resultados_64/{nombre}", f"resultados_kaggle_64/{nombre}"] + glob.glob(f"/kaggle/input/**/{nombre}", recursive=True):
        if os.path.exists(ruta):
            return ruta
    print("descargando", nombre, "del repositorio…")
    urllib.request.urlretrieve(REPO_RAW + nombre, nombre)
    return nombre

ruta_pesos = buscar(PESOS)
print("pesos:", ruta_pesos)

from medmnist import OrganMNIST3D
OrganMNIST3D(split="train", download=True, size=64)
ruta_npz = os.path.expanduser("~/.medmnist/organmnist3d_64.npz")
md5 = hashlib.md5(open(ruta_npz, "rb").read()).hexdigest()
assert md5 == MD5_64, md5
d = np.load(ruta_npz)
X_train, X_test, y_test = d["train_images"], d["test_images"], d["test_labels"].reshape(-1)
print("md5 ok |", "train", X_train.shape, "| test", X_test.shape)

## 2. La red (misma definición que en el entrenamiento)

In [ ]:
class Bloque(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv3d(cin, cout, 3, stride, 1, bias=False); self.b1 = nn.BatchNorm3d(cout)
        self.c2 = nn.Conv3d(cout, cout, 3, 1, 1, bias=False); self.b2 = nn.BatchNorm3d(cout)
        self.sc = None if (stride == 1 and cin == cout) else nn.Sequential(
            nn.Conv3d(cin, cout, 1, stride, bias=False), nn.BatchNorm3d(cout))

    def forward(self, x):
        o = F.relu(self.b1(self.c1(x)))
        o = self.b2(self.c2(o))
        return F.relu(o + (x if self.sc is None else self.sc(x)))


class ResNet3D(nn.Module):
    def __init__(self, anchos=(16, 32, 64, 128), n_clases=11):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv3d(1, anchos[0], 3, 1, 1, bias=False), nn.BatchNorm3d(anchos[0]), nn.ReLU(inplace=True))
        capas, c = [], anchos[0]
        for i, a in enumerate(anchos):
            capas += [Bloque(c, a, 1 if i == 0 else 2), Bloque(a, a, 1)]
            c = a
        self.capas = nn.Sequential(*capas)
        self.fc = nn.Linear(c, n_clases)

    def forward(self, x):
        return self.fc(self.capas(self.stem(x)).mean(dim=(2, 3, 4)))


modelo = ResNet3D()
modelo.load_state_dict(torch.load(ruta_pesos, map_location="cpu"))
modelo.eval()
print("parámetros:", sum(p.numel() for p in modelo.parameters()))

# estadísticas de normalización: solo con entrenamiento, igual que en el entrenamiento
x_tr = X_train.astype(np.float32) / 255.0
MEDIA, DESV = float(x_tr.mean()), float(x_tr.std() + 1e-8)
del x_tr
print(f"media {MEDIA:.4f}  desv {DESV:.4f}")


@torch.no_grad()
def clasificar(v):
    """v: volumen uint8 (64, 64, 64). Devuelve las 11 probabilidades."""
    x = (v.astype(np.float32) / 255.0 - MEDIA) / DESV
    x = torch.from_numpy(x)[None, None]            # (1, 1, 64, 64, 64)
    return F.softmax(modelo(x), 1)[0].numpy()

## 3. Clasificar un volumen de prueba

Cambia `INDICE` por cualquier número entre 0 y 609. Los de la Figura 5 del informe son 531, 376, 301, 138, 187, 15, 175, 5, 64, 544 y 386 (uno por clase).

In [ ]:
INDICE = 187   # fémur izquierdo: la red 3D a 28³ lo confundía con el derecho


def mostrar(indice):
    v, real = X_test[indice], int(y_test[indice])
    p = clasificar(v)
    pred = int(p.argmax())
    m = v.shape[0] // 2
    fig, ax = plt.subplots(1, 4, figsize=(13, 3.4), gridspec_kw={"width_ratios": [1, 1, 1, 1.9]})
    lo, hi = np.percentile(v, [1, 99])
    for a, corte, nombre in zip(ax[:3], (v[m], v[:, m], v[:, :, m]), ("eje 1", "eje 2", "eje 3")):
        a.imshow(corte, cmap="gray", vmin=lo, vmax=hi); a.set_title(nombre, fontsize=9); a.axis("off")
    colores = ["#1a7f37" if i == real else ("#cf222e" if i == pred else "#9aa0a6") for i in range(11)]
    ax[3].barh(range(11), p, color=colores)
    ax[3].set_yticks(range(11)); ax[3].set_yticklabels(CLASES, fontsize=8); ax[3].invert_yaxis()
    ax[3].set_xlim(0, 1); ax[3].set_xlabel("probabilidad")
    for i, pi in enumerate(p):
        if pi >= 0.01:
            ax[3].text(min(pi + 0.01, 0.86), i, f"{pi:.2f}", va="center", fontsize=8)
    ok = "acierta" if pred == real else "falla"
    fig.suptitle(f"volumen de prueba {indice}: real = {CLASES[real]} | predicción = {CLASES[pred]} ({p[pred]:.2f}) → {ok}",
                 fontsize=10, color="#1a7f37" if pred == real else "#cf222e")
    plt.tight_layout(); plt.show()
    return p


_ = mostrar(INDICE)

Verde: clase real; rojo: predicción equivocada (si la hay); gris: las demás clases.

In [ ]:
for i in (544, 376, 301):     # bazo (la 2.5D lo confundía con hígado) y los dos riñones
    _ = mostrar(i)

## 4. Comprobación sobre todo el conjunto de prueba

Debe dar la exactitud de la semilla 0 que reporta el informe: **0,957** (584 de 610).

In [ ]:
aciertos = 0
for i in range(len(y_test)):
    aciertos += int(clasificar(X_test[i]).argmax() == y_test[i])
print(f"exactitud en prueba: {aciertos / len(y_test):.3f} ({aciertos} de {len(y_test)})")